In [1]:
import pandas as pd
import torch
import torch.nn
import torch.optim as optim
from sklearn.model_selection import  train_test_split
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import  LabelEncoder
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer


In [2]:
df = pd.read_csv(r"C:\Users\pc\OneDrive\Music\Desktop\ML_Projects\Dataset\IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [4]:
df.drop_duplicates(inplace=True)
df.shape

(49582, 2)

In [5]:
# Preprocessing
import re
df["review"] = df["review"].str.lower()

In [6]:
# Remove URLs

def remove_urls(text):
    text = re.sub(r"http\S+", "", text)
    return text
df["review"] = df["review"].apply(remove_urls)

In [7]:
# Remove Punctuations
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "", text)
    return text
df["review"] = df["review"].apply(remove_punctuations)

In [8]:
# Remove HTML tags

def remove_html(text):
    text = re.sub(r"[<.*?>]", "", text)
    return text
df["review"] = df["review"].apply(remove_html)

In [9]:
# Remove Stopwords

def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")
    for word in tokens:
        text = text.replace(word, "")
    return text
df["review"] = df["review"].apply(remove_stopwords)

In [10]:
# Stemming

def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []
    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)
    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [11]:
# Encoding & Vectorization

le = LabelEncoder()
y = df["sentiment"] = le.fit_transform(df["sentiment"])
df["sentiment"].value_counts()

sentiment
1    24884
0    24698
Name: count, dtype: int64

In [12]:
# TF-IDF Vectorization

tf = TfidfVectorizer(max_features=5000)
x = tf.fit_transform(df["review"])

In [13]:
# Train Test Split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)
x_train.shape, x_test.shape

((39665, 5000), (9917, 5000))

In [14]:
# Convert TensorDataset
x_train = x_train.toarray()
x_test  = x_test.toarray()

In [15]:
# Convert

train_set = TensorDataset(
    torch.from_numpy(x_train).float(),
    torch.from_numpy(y_train).float()
)

test_set = TensorDataset(
    torch.from_numpy(x_test).float(),
    torch.from_numpy(y_test).float()
)

In [16]:
#  Dataloader 

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_set, batch_size=64, shuffle=True)

In [17]:
import torch.nn as nn

class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0) 
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

In [18]:
input_size = x_train.shape[1]

model = RNN(input_size)

criteria = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

In [19]:
# Training Model

epochs = 20

for epoch in range (epochs):
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        
        xb = xb.unsqueeze(1) # Adding Singleton direction
        output = model(xb)   # Forward Propagation..
        output = torch.sigmoid(output.squeeze())

        loss = criteria(output, yb) # Comput Loss
        loss.backward()     # Backward Propagation
        optimizer.step()        # Update Weight

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/20 and loss = 0.4042687714099884
epoch = 2/20 and loss = 0.22111712396144867
epoch = 3/20 and loss = 0.28114232420921326
epoch = 4/20 and loss = 0.36026322841644287
epoch = 5/20 and loss = 0.3877450227737427
epoch = 6/20 and loss = 0.3417612612247467
epoch = 7/20 and loss = 0.2763740122318268
epoch = 8/20 and loss = 0.2526082694530487
epoch = 9/20 and loss = 0.2861907482147217
epoch = 10/20 and loss = 0.291923463344574
epoch = 11/20 and loss = 0.33207300305366516
epoch = 12/20 and loss = 0.4084431529045105
epoch = 13/20 and loss = 0.2229120284318924
epoch = 14/20 and loss = 0.4031018912792206
epoch = 15/20 and loss = 0.5196871757507324
epoch = 16/20 and loss = 0.32467591762542725
epoch = 17/20 and loss = 0.4484271705150604
epoch = 18/20 and loss = 0.45533081889152527
epoch = 19/20 and loss = 0.24580974876880646
epoch = 20/20 and loss = 0.41232019662857056


In [25]:
# Validation

model.eval()

val_loss = 0
correct = 0
total = 0

with torch.no_grad():

    for xb, yb in test_loader:
        xb = xb.unsqueeze(1)
        output = model(xb)  # Forward Propagation
        output = torch.sigmoid(output.squeeze())
        loss = criteria(output, yb)
        val_loss += loss.item()
        predicted = (output > 0.5).float()
        total += yb.size(0)
        correct += (predicted == yb).sum().item() 
val_loss = val_loss / len(test_loader)
accuracy = correct / total

print(f"Validation Loss:{val_loss}")
print(f"Validation Accuracy: {accuracy*100}")

Validation Loss:0.42181061477430404
Validation Accuracy: 81.53675506705656


In [20]:
# Evaluate 

model.eval()

with torch.no_grad():
    correct_vals = 0
    total = 0

    for xb, yb in test_loader:

        xb = xb.unsqueeze(1)

        output = model(xb)      # Forward Propagation
        predicted = (torch.sigmoid(output.squeeze())> 0.5).float()

        total += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"Accuarcy = {correct_vals/total*100}")

Accuarcy = 81.53675506705656
